In [52]:
import pandas as pd
import numpy as np
import krippendorff as kd

## Remark: this notebook was used to iteratively develop the majority/priority based datasets for final testing and evaluation and as no further usage

# sort pathologists by mean performance (updated in final version)

In [ ]:
# avg_alpha_path = []
# for path in range (1,21):
#     alpha_per_fold = []
#     for fold in range(1,6):
#         df = pd.read_csv(f"../../../experiments/intra/evaluation_results_final_intra_path_{path}_fold_{fold}.csv")
#         alpha = kd.alpha(df[["pred_class", "label"]].T.values, level_of_measurement="ordinal", value_domain=[0, 1.0, 2.0])
#         alpha_per_fold.append(alpha)
#     avg_alpha_path.append(np.mean(alpha_per_fold))
# results = pd.DataFrame({"path_id": list(range(1, 21)), "alpha": avg_alpha_path})
# results.to_csv("../../../experiments/intra/krippendorff_alpha_per_path.csv", index=False)

# create labels based on mode/rater priority

In [54]:
avg_alpha = pd.read_csv("../../../experiments/intra/krippendorff_alpha_per_path.csv")
labels = pd.read_csv('EDA/data/lans_all_labels.csv')
cons = labels[["block_id", "dx"]]
cons = cons[["block_id", "dx"]].replace({1: 0, 2: np.nan, 3: 1, 4: 2})
cons = cons.dropna(subset=["dx"])
rater_labels = labels.loc[:,"p53":]
rater_labels = rater_labels.drop(["p53"], axis=1).replace({1: 0, 2: np.nan, 3: 1, 4: 2, 0: np.nan})
rater_labels = pd.concat([labels["block_id"], rater_labels], axis=1)

In [55]:
path_list = [1, 3, 4, 2, 5]
path_scores = avg_alpha[avg_alpha["path_id"].isin(path_list)][["path_id", "alpha"]].sort_values(by="alpha", ascending=False)
path_list_priority = path_scores["path_id"].tolist()
col_list = [f"path_{i}" for i in path_list]

In [56]:
modes = rater_labels[col_list].mode(axis=1, numeric_only=True)
num_different_modes = modes.notna().sum(axis=1)
undecided_cases_indices = num_different_modes[num_different_modes > 1].index

In [62]:
decided_cases_indices = num_different_modes[num_different_modes == 1].index
decided_df = modes[modes.index.isin(decided_cases_indices)][[0]].rename(columns={0: "labels"})
decided_df["block_id"] = rater_labels.loc[decided_cases_indices, "block_id"].values
decided_df

,labels,block_id
0,0.0,RL-0001-I
2,0.0,RL-0001-III
3,0.0,RL-0002-I
4,1.0,RL-0003-II
5,2.0,RL-0004-I
...,...,...
2069,1.0,RL-1098-I
2070,2.0,RL-1098-II
2071,1.0,RL-1099-I
2072,2.0,RL-1100-I


In [63]:
undecided_cases_basis = rater_labels[rater_labels.index.isin(undecided_cases_indices)]
undecided_df = pd.DataFrame({"block_id": undecided_cases_basis["block_id"]})

undecided_labels = []
for index, row in undecided_cases_basis.iterrows():
    for path in path_list_priority:
        print(f"Processing block_id: {row['block_id']}")
        col = f"path_{path}"
        cell_value = row[f"path_{path}"]
        print(f"Checking {col}: {cell_value}")
        if pd.notna(cell_value):
            undecided_labels.append(cell_value)
            print(f"Selected {cell_value} from {col}")
            break
    else:
       undecided_labels.append(np.nan) 

undecided_df["labels"] = undecided_labels


Processing block_id: RL-0001-II
Checking path_2: nan
Processing block_id: RL-0001-II
Checking path_4: 0.0
Selected 0.0 from path_4
Processing block_id: RL-0015-II
Checking path_2: 0.0
Selected 0.0 from path_2
Processing block_id: RL-0015-III
Checking path_2: 1.0
Selected 1.0 from path_2
Processing block_id: RL-0033-V
Checking path_2: nan
Processing block_id: RL-0033-V
Checking path_4: 0.0
Selected 0.0 from path_4
Processing block_id: RL-0037-III
Checking path_2: 1.0
Selected 1.0 from path_2
Processing block_id: RL-0041-I
Checking path_2: nan
Processing block_id: RL-0041-I
Checking path_4: nan
Processing block_id: RL-0041-I
Checking path_3: nan
Processing block_id: RL-0041-I
Checking path_5: 1.0
Selected 1.0 from path_5
Processing block_id: RL-0043-I
Checking path_2: nan
Processing block_id: RL-0043-I
Checking path_4: 1.0
Selected 1.0 from path_4
Processing block_id: RL-0055-II
Checking path_2: 0.0
Selected 0.0 from path_2
Processing block_id: RL-0061-III
Checking path_2: 1.0
Selected 1

In [64]:
undecided_df

,block_id,labels
1,RL-0001-II,0.0
33,RL-0015-II,0.0
34,RL-0015-III,1.0
88,RL-0033-V,0.0
97,RL-0037-III,1.0
...,...,...
2000,RL-1060-I,1.0
2002,RL-1061-II,1.0
2024,RL-1076-VII,2.0
2028,RL-1079-II,1.0


In [68]:
pd.concat([undecided_df, decided_df], axis=0).sort_index()

,block_id,labels
0,RL-0001-I,0.0
1,RL-0001-II,0.0
2,RL-0001-III,0.0
3,RL-0002-I,0.0
4,RL-0003-II,1.0
...,...,...
2070,RL-1098-II,2.0
2071,RL-1099-I,1.0
2072,RL-1100-I,2.0
2073,RL-1100-II,1.0
